# Retrieval-Augmented Generation (RAG)

The #1 use case for Large Language Models (LLMs) today is putting them over internal documents to make information in those documents easily discoverable. Retrieval-augmented generation (RAG) is a technique that allows LLMs to work with any number of documents of any size. LLMs limit how much text can be input to them in a single call. RAG involves dividing documents into "chunks" of (typically) a few hundred words each and generating an embedding vector for each chunk. To answer a question, you generate an embedding vector from the question, identify the *n* most similar embedding vectors, and provide the corresponding chunks of text to the LLM. To improve results, you can use a *reranker* to determine which chunks are the most relevant. Let's demonstrate with a CSV file containing more than 8,000 reviews of BMWs.

Start by creating a ChromaDB database to hold the reviews:

In [1]:
import chromadb
from chromadb.config import Settings

# This is still the best layout to keep for your settings
client = chromadb.PersistentClient(
    path='chroma',
    settings=Settings(anonymized_telemetry=False)
)

collection = client.get_or_create_collection(name='Car_Reviews')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Now read the CSV file, concatenate the vehicle title and review in each row, and insert the resulting text into the database. Embedding vectors are generated automatically by ChhromaDB. Note that this code crashes the kernel with some versions of ChromaDB. Crashes can usually be averted by downgrading the Python package named `chroma-hnswlib` to version 0.7.3 — for example `pip install chroma-hnswlib==0.7.3`.

In [2]:
import re
import pandas as pd

BATCH_SIZE = 100
documents = []
ids = []

df = pd.read_csv('Data/bmw.csv', engine='python')

# Remove rows with empty vehicle titles or reviews
df = df.dropna(subset=['Vehicle_Title', 'Review'])

for i, row in df.iterrows():
    vehicle = row['Vehicle_Title']
    review = row['Review']

    if vehicle and len(vehicle) > 8 and review and len(review) > 128:
        review = re.sub(r'[\r\n\t]+', ' ', review)
        review = re.sub(r'\s{2,}', ' ', review)
        review = review.strip()
        
        text = f'Model: {vehicle}\nReview: {review}'
        documents.append(text)
        ids.append(f'{i:05}')
        
        if (i + 1) % BATCH_SIZE == 0 and len(documents) > 0:
            collection.add(documents=documents, ids=ids)
            documents = []
            ids = []

if len(documents) > 0:
    collection.add(documents=documents, ids=ids)

Insert of existing embedding ID: 00001
Insert of existing embedding ID: 00003
Insert of existing embedding ID: 00005
Insert of existing embedding ID: 00006
Insert of existing embedding ID: 00007
Insert of existing embedding ID: 00009
Insert of existing embedding ID: 00010
Insert of existing embedding ID: 00011
Insert of existing embedding ID: 00012
Insert of existing embedding ID: 00013
Insert of existing embedding ID: 00014
Insert of existing embedding ID: 00015
Insert of existing embedding ID: 00016
Insert of existing embedding ID: 00017
Insert of existing embedding ID: 00018
Insert of existing embedding ID: 00019
Insert of existing embedding ID: 00020
Insert of existing embedding ID: 00021
Insert of existing embedding ID: 00022
Insert of existing embedding ID: 00023
Insert of existing embedding ID: 00024
Insert of existing embedding ID: 00026
Insert of existing embedding ID: 00027
Insert of existing embedding ID: 00028
Insert of existing embedding ID: 00029
Insert of existing embedd

Show the first 5 items added to the database:

In [3]:
items = collection.get()

for document in items['documents'][:5]:
    print(document)
    print('-' * 40)

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Model: 2014 BMW X1 SUV xDrive28i 4dr SUV AWD (2.0L 4cyl Turbo 8A)
Review: We picked this car as a CPO from a BMW dealer. No problems to date. Relatively decent handling for the type car, and just the right size and height. We stayed away from the newer version to avoid a Mini based front wheel platform derived car as opposed to a BMW based one. The new 2018 X1 also features a flimsy cloth covering under the moonroof that I can see ripping quickly. If you want a X1 like this may I suggest springing for the upgraded interior. Base level seating has poor lateral support and are unacceptable. Base halogen headlights are unacceptable and should be avoided. My 3 series coupe has xenons which are "light years" better. Sorry!:-) Base sound system is "media-ocre" at best. I don't understand the need for moonroofs on BMWs, and wish they would apportion the monies elsewhere and delete them. Acceleration with the 240 hp is considerably better than you would expect for a 4 cyl. It feels like a 6! B

Query the database and show the top 5 items that are likely to contain an answer to a question:

In [4]:
results = collection.query(
    query_texts=['How reliable are BMWs? Are they expensive to work on?'],
    n_results=5
)

documents = list(reversed(results['documents'][0]))
scores = list(reversed(results['distances'][0]))

for index, document in enumerate(documents):
    print(f'Score: {scores[index]}')
    print(document)
    print('-' * 40)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Score: 0.786305844783783
Model: 2002 BMW 3 Series Coupe 330Ci Rwd 2dr Coupe (3.0L 6cyl 5M)
Review: Incredible in every way! After owning many BMWs this is no disappointment. My last was an E36 99 M3. BMW has come a long way in terms of refinement, but build quality has suffered a bit compared to the E36 cars, especially interior. It is nowhere as "raw" as the previous generation, which some owners may see as regression. Smooth torqueband, perfect styling, handling hardwired to your brain. This car is really so much fun to drve and like any other BMW, you love it more and more every day. Reliability is perfect and even maintenance parts are dirt cheap compared to Honda, Toyota, etc. Just know where to look!
----------------------------------------
Score: 0.7800027132034302
Model: 1999 BMW 3 Series Sedan 323i 4dr Sedan
Review: After owning this car for over 5 years I can honestly say it is the most unreliable & expensive to repair vehicle I've ever had. Everything that could go wrong has

## Use an LLM to answer questions

The next step is to use the database to retrieve relevant chunks and pass them to an LLM for question answering. Define a function that accepts a question, retrieves the 20 most relevant chunks from the database, and passes the question and the chunks to `GPT-4o`:

In [5]:
from openai import OpenAI

openai_client = OpenAI()

def answer_question(question):
    results = collection.query(
        query_texts=[question],
        n_results=20
    )

    documents = results['documents'][0]
    context = '\n\n'.join(documents)

    content = f'''
        Answer the following question using the provided context, and if the
        answer is not contained within the context, say "I don't know." Explain
        your answer if possible. Do not mention the provided context in your
        output. Do not use markdown formatting.
        
        Question:
        {question}

        Context:
        {context}
        '''

    messages = [{ 'role': 'user', 'content': content }]

    response = openai_client.chat.completions.create(
        model='gpt-4o',
        messages=messages,
        stream=True
    )

    # Fixed: added [0] index to choices
    for chunk in response:
        content = chunk.choices[0].delta.content
        if content is not None:
            print(content, end='')


Ask the LLM a question about BMWs:

In [6]:
answer_question('How reliable are BMWs? Are they expensive to work on?')

BMWs vary in reliability depending on the model and maintenance. Many reviews mention issues with reliability, particularly as the cars age, with complaints about mechanical failures and the cost of repairs. However, some owners report good experiences with maintenance when it's conducted regularly and thoroughly. The cost to work on BMWs is generally high, with expenses for parts and repairs often noted as being significantly more than for some other brands. Proper care and maintenance can improve reliability, but the high cost of upkeep is a common concern shared by many owners.

Ask another question:

In [7]:
answer_question('What are three good reasons to buy a BMW? What are three good reasons NOT to buy a BMW?')

Three good reasons to buy a BMW could be its superior driving dynamics, luxury features, and strong handling capabilities. BMWs are often praised for their engaging driving experience, providing a feeling of connection to the road that many drivers find exhilarating. Additionally, the luxury and comfort offered by BMW interiors are often highlighted, making them appealing for those looking for a refined driving experience.

Three good reasons not to buy a BMW might include high maintenance costs, reliability issues, and complex electronic systems. Many reviews suggest that maintenance for BMWs can be quite expensive, and there may be reliability concerns with certain models. The complexity of the electronics can also be frustrating for some users, making the vehicles less user-friendly than competitors.

Make sure the LLM will answer "I don't know" to a question that has nothing to do with BMWs:

In [8]:
answer_question('Why is the sky blue?')

I don't know.

## Add reranking

Using the distance between embedding vectors to determine relevance isn't perfect. It's frequently helpful to retrieve candidate chunks from the database and then use a *reranker* to determine which chunks really are the most relevant. Rerankers come in many forms. The most commonly used rerankers are *cross encoders*, which are language models trained to quantify the relevance between two text samples. Let's load the `jina-reranker-v1-turbo-en` cross encoder from [Hugging Face](https://huggingface.co/jinaai/jina-reranker-v1-turbo-en) so we can use it for reranking:

In [9]:
from sentence_transformers import CrossEncoder

model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', trust_remote_code=True)

C:\Users\sumida\anaconda3\envs\GenerativeAI\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3813.60it/s]


Retrieve 10 chunks from the database to answer a question, and then use the reranker to pick the top 5. How do these chunks compare to 5 selected earlier?

In [10]:
question = 'How reliable are BMWs? Are they expensive to work on?'

results = collection.query(
    query_texts=[question],
    n_results=10
)

documents = results['documents'][0]
ranked_documents = model.rank(question, documents, return_documents=True, top_k=5)

for document in ranked_documents:
    print(f"Score: {document['score']}")
    print(document['text'])
    print('-' * 40)

Score: 4.110507488250732
Model: 1999 BMW 3 Series Sedan 323i 4dr Sedan
Review: After owning this car for over 5 years I can honestly say it is the most unreliable & expensive to repair vehicle I've ever had. Everything that could go wrong has and at a huge cost to repair. Anyone thinking of buying one of these cars should think again! I've endured complete brake replacements every 15K miles, had to replace almost the entire front suspension (worn ball joints can not be replaced by themselves), the mass air flow sensor, all 6 ignition coils, cam shaft sensor and a whole host of other "should be reliable" electronic components have failed. If someone gives you one of these cars - sell it!
----------------------------------------
Score: 3.6830830574035645
Model: 2008 BMW 5 Series Sedan 528xi 4dr Sedan AWD (3.0L 6cyl 6M)
Review: Car is not reliable at all. High maintenance cost. Frequent mechanical visits. Parts expensive. Rear leg room is terrible. Brake are loud & dusty. Electric issues 

Rewrite the `answer_question` function to retrieve 40 chunks from the database and use the cross encoder to pick the best 20:

In [11]:
def answer_question(question):
    # This automatically picks up the global 'collection' variable from your first cell
    results = collection.query(
        query_texts=[question],
        n_results=20
    )

    documents = results['documents'][0]
    context = '\n\n'.join(documents)

    content = f'''
        Answer the following question using the provided context, and if the
        answer is not contained within the context, say "I don't know." Explain
        your answer if possible. Do not mention the provided context in your
        output. Do not use markdown formatting.
        
        Question:
        {question}

        Context:
        {context}
        '''

    messages = [{ 'role': 'user', 'content': content }]

    # Use whatever global variable name you chose for OpenAI in your earlier cell here:
    response = openai_client.chat.completions.create(
        model='gpt-4o',
        messages=messages,
        stream=True
    )

    for chunk in response:
        content = chunk.choices[0].delta.content
        if content is not None:
            print(content, end='')


Use the modified function to answer a question about BMWs:

In [12]:
answer_question('How reliable are BMWs? Are they expensive to work on?')

BMWs have a mixed reputation regarding reliability, with some models and years showing significant reliability issues while others perform well with regular maintenance. Generally, BMWs can be expensive to work on, with costs for parts and repairs being higher than average due to the brand's premium nature. Maintenance costs are often noted to be pricey, and specific models may have known issues requiring attention. Proper care and maintenance can help mitigate some reliability concerns, but overall, they are considered to have higher maintenance costs compared to some competitors, particularly Japanese brands.

Try again with a question that the LLM wasn't able to answer earlier:

In [13]:
answer_question('What are three good reasons to buy a BMW? What are three good reasons NOT to buy a BMW?')

Three good reasons to buy a BMW could be its excellent driving experience, refined build quality, and stylish design. On the other hand, three reasons not to buy a BMW might include high maintenance costs, potential reliability issues, and complex electronic systems that can be difficult to manage.

Make sure the LLM will still admit it doesn't know the answer when the chunks provided to it don't contain the answer to a question:

In [14]:
answer_question('Why is the sky blue?')

I don't know.